# EFT Limit Extraction with Toys

Demonstrates Wilson coefficient limits using asymptotic and toy-based hypothesis testing.

**Model:** $\sigma(c) = \sigma_{SM} \cdot (1 + Ac + Bc^2)$

**Architecture:** This notebook demonstrates the separation of concerns:
- **Test Statistics** (`QTilde`): Compute likelihood ratios
- **Distributions** (`QTildeAsymptotic`, `EmpiricalDistribution`): Convert q → p-values
- **Calculator** (`HypoTestCalculator`): Orchestrates the workflow
- **ToyGenerator**: Creates empirical distributions from Monte Carlo

In [1]:
import jax
import jax.numpy as jnp
from jax.scipy.special import gammaln

import everwillow.statelib as sl
from everwillow.inference.hypotest import (
    HypoTestCalculator,
    QTilde,
    QTildeAsymptotic,
    ToyGenerator,
    expected_upper_limit,
    upper_limit,
    upper_limit_toys,
)

jax.config.update("jax_enable_x64", True)

## Model Definition

In [2]:
# EFT parameters
SM_PRED = 100.0  # SM prediction at c=0
A = 0.5          # Linear (interference) term
B = 0.1          # Quadratic term
ntoys = 10000    # Number of toys for toy-based limit


def make_nll_factory(sm_pred: float, A: float, B: float):
    """NLL factory for single-bin Poisson measurement."""
    def nll_factory(params_state: sl.State, observed: dict):
        def nll_fn(params: dict) -> float:
            c = params["c"]
            pred = sm_pred * (1.0 + c * A + c**2 * B)
            pred = jnp.maximum(pred, 1e-10)
            obs = observed["main"]
            return pred - obs * jnp.log(pred) + gammaln(obs + 1)
        return nll_fn
    return nll_factory


def make_sample_fn(sm_pred: float, A: float, B: float):
    """Sampling function for toy generation."""
    def sample_fn(params_state: sl.State, key):
        c = params_state.to_pytree()["c"]
        expected = sm_pred * (1.0 + c * A + c**2 * B)
        return {"main": jax.random.poisson(key, jnp.maximum(expected, 0.0))}
    return sample_fn

## Setup

In [3]:
# Observed data
observed = {"main": 100.0}

# Create functions
nll_factory = make_nll_factory(SM_PRED, A, B)
sample_fn = make_sample_fn(SM_PRED, A, B)
nll_fn = lambda params: nll_factory(None, observed)(params)

# Initial parameters
params = sl.State.from_pytree({"c": 0.0})

# Calculator and distributions (separation of concerns!)
calc = HypoTestCalculator(test_statistic=QTilde())
asymp_dist = QTildeAsymptotic()
toy_gen = ToyGenerator(test_statistic=QTilde(), ntoys=ntoys)
key = jax.random.key(42)

print(f"Model: σ/σ_SM = 1 + {A}c + {B}c²")
print(f"Observed: {observed['main']:.0f} events | SM pred: {SM_PRED:.0f} | Toys: {ntoys}")

Model: σ/σ_SM = 1 + 0.5c + 0.1c²
Observed: 100 events | SM pred: 100 | Toys: 10000


## CLs Comparison

In [4]:
print("c\t Asymp\t  Toys")
print("-" * 25)
for i, c_test in enumerate([0.0, 0.1, 0.3, 0.5, 0.7, 1.0]):
    # Asymptotic: use QTildeAsymptotic distribution
    result_a = calc(nll_fn, params, ("c",), c_test, distribution=asymp_dist)

    # Toy-based: generate EmpiricalDistribution then use calculator
    emp_dist = toy_gen.generate(
        nll_fn, params, ("c",), c_test,
        sample_fn=sample_fn, nll_factory=nll_factory,
        key=jax.random.fold_in(key, i)
    )
    result_t = calc(nll_fn, params, ("c",), c_test, distribution=emp_dist)

    print(f"{c_test:.1f}\t {float(result_a.cl_s):.4f}\t {float(result_t.cl_s):.4f}")

c	 Asymp	  Toys
-------------------------


0.0	 1.0000	 1.0000


0.1	 0.6160	 0.6334


0.3	 0.1303	 0.1374


0.5	 0.0113	 0.0128


0.7	 0.0004	 0.0004


1.0	 0.0000	 0.0000


## Upper Limits (95% CL)

In [5]:
# Asymptotic limit
limit_asymp = upper_limit(
    lambda poi: calc(nll_fn, params, ("c",), poi, distribution=asymp_dist).palt,
    bounds=(0.0, 2.0),
    level=0.05
)
print(f"Asymptotic: c < {float(limit_asymp):.3f}")

Asymptotic: c < 0.326


In [6]:
# Toy-based limit
limit_toys = upper_limit_toys(
    lambda poi, k: calc(
        nll_fn, params, ("c",), poi,
        distribution=toy_gen.generate(
            nll_fn, params, ("c",), poi,
            sample_fn=sample_fn, nll_factory=nll_factory, key=k
        )
    ).palt,
    bounds=(0.0, 2.0),
    key=key,
    level=0.05
)
print(f"Toys:       c < {float(limit_toys):.3f}")

Toys:       c < 0.359


## Expected Limit Bands

Expected limits under background-only hypothesis (Brazil bands).

In [7]:
# Compute all limits with Brazil bands in one call
limits = expected_upper_limit(
    lambda poi: calc(nll_fn, params, ("c",), poi, distribution=asymp_dist),
    bounds=(-100.0, 100.0),
)

print("95% CL Upper Limits (Brazil bands):")
print(f"  -2σ: c < {float(limits.minus_2sigma):.3f}")
print(f"  -1σ: c < {float(limits.minus_1sigma):.3f}")
print(f"  Exp: c < {float(limits.expected):.3f}")
print(f"  +1σ: c < {float(limits.plus_1sigma):.3f}")
print(f"  +2σ: c < {float(limits.plus_2sigma):.3f}")
print(f"  Obs: c < {float(limits.observed):.3f}")

95% CL Upper Limits (Brazil bands):
  -2σ: c < 0.209
  -1σ: c < 0.280
  Exp: c < 0.388
  +1σ: c < 0.538
  +2σ: c < 0.719
  Obs: c < 0.388


## JIT Compilation Timing

In [8]:
import time

@jax.jit
def compute_limit_jit(key):
    return upper_limit_toys(
        lambda poi, k: calc(
            nll_fn, params, ("c",), poi,
            distribution=toy_gen.generate(
                nll_fn, params, ("c",), poi,
                sample_fn=sample_fn, nll_factory=nll_factory, key=k
            )
        ).palt,
        bounds=(0.0, 2.0),
        key=key,
        level=0.05
    )

# First call includes compilation
start = time.time()
limit1 = compute_limit_jit(jax.random.key(1))
t_compile = time.time() - start

# Second call uses cached compilation
start = time.time()
limit2 = compute_limit_jit(jax.random.key(1))
t_cached = time.time() - start

print(f"First call (compile): {t_compile:.2f}s → c < {float(limit1):.3f}")
print(f"Second call (cached): {t_cached:.2f}s → c < {float(limit2):.3f}")
print(f"Speedup: {t_compile/t_cached:.1f}x")

First call (compile): 0.85s → c < 0.344
Second call (cached): 0.08s → c < 0.344
Speedup: 11.3x
